# How many failure types are still undiscovered?

This notebook runs the pilot on Colab using every CPU core in parallel. No GPU is needed: choose **Runtime → Change runtime type → CPU** (High-RAM if available).

Run the cells in order. The pilot (3 datasets × 3 models × 5 seeds) takes about 10–20 minutes on 8 cores.

In [1]:
# 1. Get the code and check it
!git clone -q https://github.com/ParitKansal/undiscovered-failures.git
%cd undiscovered-failures
!pip install -q -r requirements.txt
!python -m tests.test_estimators
!python -m tests.test_end_to_end | tail -1
import os; print('CPU cores:', os.cpu_count())

/content/undiscovered-failures
ALL 16 ESTIMATOR CHECKS PASSED
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   8 out of   8 | elapsed:    4.7s finished
END-TO-END TEST PASSED
CPU cores: 24


In [2]:
# 2. Optional: save results to Google Drive so they survive a disconnect
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/undiscovered-failures'
os.makedirs(OUT, exist_ok=True)

Mounted at /content/drive


In [ ]:
# 3. Run the pilot in parallel (all cores)
!python experiment.py --datasets adult,bank,covertype --models logreg,forest,boosting --seeds 5 --repeats 30 --out {OUT}/pilot.csv --data_home /content/data

In [ ]:
# 4. Summary tables
!python summarize.py {OUT}/pilot.csv

## Pilot v1: wider design and bootstrap intervals
4 datasets (adds MAGIC) × 3 models × 3 seeds × 2 flip rates (0.6, 0.9) × 3 numbers of planted types (15, 30, 60) = 216 jobs.
Each row now also has bootstrap 95% intervals for Chao1, ACE and the jackknife. About 30–40 minutes on 8 cores (faster with more).

In [ ]:
# 5. Run pilot v1 (all cores)
!git pull -q
!python -m tests.test_estimators
!python experiment.py --datasets adult,bank,covertype,magic --models logreg,forest,boosting --seeds 3 --repeats 30 --flips 0.6,0.9 --k_planted 15,30,60 --boot 200 --out {OUT}/pilot_v1.csv --data_home /content/data

In [ ]:
# 6. Summary tables for v1
!python summarize.py {OUT}/pilot_v1.csv

## Pilot v2: bias-calibrated intervals and the unseen share of errors
Same design as v1. New columns: a bias-calibrated bootstrap (estimate and interval), and the share of all errors that belong to types not yet seen, true versus estimated (1 - sample coverage). A few minutes on 24 cores.

In [ ]:
# 7. Run pilot v2 (all cores)
!git pull -q
!python -m tests.test_estimators
!python experiment.py --datasets adult,bank,covertype,magic --models logreg,forest,boosting --seeds 3 --repeats 30 --flips 0.6,0.9 --k_planted 15,30,60 --boot 200 --out {OUT}/pilot_v2.csv --data_home /content/data

In [ ]:
# 8. Summary tables for v2 (the first table holds the key numbers)
!python summarize.py {OUT}/pilot_v2.csv

In [ ]:
# 8b. Calibration: typical (absolute) error of the unseen-share estimate in pilot v2, to set the pre-registered thresholds
!git pull -q
!python calibrate_from_pilot.py {OUT}/pilot_v2.csv

## Main study (pre-registered, see PREREG.md)
Run only after PREREG.md is committed. Fresh seeds 100-104 (never used in the pilots); 5 datasets × 3 models × 5 seeds × 2 flip rates × 3 numbers of planted types = 450 jobs, each also training a clean model for natural failures. About 10-15 minutes on 24 cores.

In [3]:
# 9. Main study (all cores)
!git pull -q
!python -m tests.test_estimators
!python experiment.py --datasets adult,bank,covertype,magic,electricity --models logreg,forest,boosting --seeds 5 --seed_offset 100 --repeats 30 --flips 0.6,0.9 --k_planted 15,30,60 --boot 200 --out {OUT}/main.csv --data_home /content/data

ALL 16 ESTIMATOR CHECKS PASSED
450 jobs on 24 CPU cores
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:   53.0s
/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[Parallel(n_jobs=-1)]: Done 114 tasks      | elapsed:  3.7min
[Parallel(n_jobs=-1)]: Done 240 tasks      | elapsed:  7.9min
[Parallel(n_jobs=-1)]: Done 402 tasks      | elapsed: 10.7min
[Parallel(n_jobs=-1)]: Done 450 out of 450 | elapsed: 11.3min finished
wrote /content/drive/MyDrive/undiscovered-failures/main.csv 283500 rows


In [4]:
# 10. Main analysis: the pre-registered decisions, then the summary tables
!python analyze_main.py {OUT}/main.csv
!python summarize.py {OUT}/main.csv

162000 rows, 180 cells

== PLANTED (45 cells)
H1 unseen share unbiased, |mean signed error| <= 0.04: 95.6% of cells -> SUPPORTED (needs >= 90%)
H2 unseen-share interval coverage >= 0.90: 71.1% of cells -> NOT SUPPORTED (needs >= 80%)
H3 ACE closer to the truth than the raw count: 88.9% of cells -> NOT SUPPORTED (needs >= 90%)
H4 (reported) bias-calibrated ACE interval coverage by budget: {300: 0.648, 1000: 0.769, 3000: 0.836}
H5 (reported) Chao-Jost beats Turing f1/n on mean |error|: 71.1% of cells
Reported: typical error per test (cell MAE) by budget: {300: 0.146, 1000: 0.05, 3000: 0.011}
Worst cells for H1:
dataset    model   types  budget  bias   mae  cov_unseen
  adult   logreg planted     300 0.076 0.265       0.703
  adult   forest planted     300 0.050 0.203       0.783
  adult boosting planted     300 0.035 0.147       0.840
  magic   logreg planted    1000 0.015 0.068       0.953
  magic boosting planted    1000 0.013 0.060       0.946

== NATURAL (135 cells)
H1 unseen share u